# 06 · Train neural networks (plan steps 3.3, 4.1–4.2)

One notebook for every neural model: list the configs in `CONFIGS` and run all; they train one after another.
Select a **T4 GPU** runtime.

| Stage | `CONFIGS` |
|---|---|
| 3 · CNN from scratch (D-012) | `["configs/cnn.yaml"]` (done) |
| 4 · MobileNetV2 (D-016, D-017) | head, ft, ft_lr3e-5 (done), then ft_lr1e-4 and ft_all: the default below |

The fine-tuning runs start from the best `mobilenetv2_head` checkpoint on Drive, so keep that order.

What happens (`plantdx.train`): the `tf.data` pipeline from `configs/base.yaml` (train cached on local disk,
augmentation on the GPU as the first model layers) → training with checkpoint on **val macro-F1** and early
stopping / learning-rate decay on **val loss** → the best epoch is scored on val → one row in the results table.

Everything for a run is written to Drive (`artifacts/runs/<run_id>/`): the training checkpoint `best.keras` and
`inference.keras` (float32, no augmentation or optimizer state, D-014). **If the session resets, run all again:**
configs whose run already finished are skipped (`SKIP_DONE`). The first epoch of a session is slower: it decodes
all JPEGs into the disk cache.

Measured on a T4: CNN about 4 min per epoch (107 min); MobileNetV2 about 75 s per epoch with a frozen backbone
and 80 s when fine-tuning (each 10-epoch run 14–15 min). The 30-epoch variants take up to about 40 min each.
Keep the tab open and the computer awake.

## 1 · Setup

In [ ]:
import os, sys
from google.colab import drive

drive.mount("/content/drive")
REPO_URL = "https://github.com/AnarHummatli/plant-disease-detection.git"
REPO_DIR = "/content/plant-disease-detection"
if os.path.isdir(f"{REPO_DIR}/.git"):
    # the Colab copy is disposable: always match GitHub exactly (D-009)
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} reset -q --hard origin/main
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
%pip install -q -e ".[dev]"
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
os.environ["PLANTDX_ARTIFACTS"] = "/content/drive/MyDrive/plantdx/artifacts"
head = !git -C {REPO_DIR} rev-parse HEAD
remote = !git -C {REPO_DIR} rev-parse origin/main
assert head == remote, "repo is not at the latest commit: run this cell again"
!git -C {REPO_DIR} log -1 --oneline
!nvidia-smi -L || echo "no GPU"
!python -m plantdx.env

## 2 · Images

In [ ]:
from plantdx.data.images import download_archive, extract_variant

VARIANT, DATA_ROOT = "color", "/content/data"
print("new files extracted:", extract_variant(download_archive(), DATA_ROOT, VARIANT))

## 3 · Train

In [ ]:
CONFIGS = [
    "configs/mobilenetv2_head.yaml",
    "configs/mobilenetv2_ft.yaml",
    "configs/mobilenetv2_ft_lr3e-5.yaml",
    "configs/mobilenetv2_ft_lr1e-4.yaml",  # D-017 variant A
    "configs/mobilenetv2_ft_all.yaml",     # D-017 variant B
]
SKIP_DONE = True  # skip configs whose run already has metrics_val.json on Drive

import json
from plantdx.config import load_config
from plantdx.paths import run_dir
from plantdx.train import train

reports = {}
for config in CONFIGS:
    run_id = load_config(config)["run_id"]
    done = run_dir(run_id, create=False) / "metrics_val.json"
    if SKIP_DONE and done.exists():
        print(f"{run_id}: already trained, skipped")
        reports[run_id] = json.loads(done.read_text())
        continue
    print(f"\n========== {run_id} ({config}) ==========")
    reports[run_id] = train(config, DATA_ROOT)
RUN_IDS = list(reports)

## 4 · Summary and learning curves ← **look at these figures**

Dotted line: the epoch kept (best val macro-F1). Check that val loss does not climb steadily while train loss
falls (overfitting), and that the curves have flattened rather than being cut off early.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image as IPImage, display

from plantdx.viz import plot_learning_curves

rows = [
    {"run_id": r, **rep["metrics"], "trainable_params": rep.get("trainable_params"),
     "size_mb": rep.get("size_mb"), "best_epoch": rep.get("best_epoch"),
     "epochs": rep.get("epochs_trained"), "minutes": round(rep.get("train_seconds", 0) / 60, 1)}
    for r, rep in reports.items()
]
display(pd.DataFrame(rows)[["run_id", "macro_f1", "accuracy", "top3", "trainable_params", "size_mb",
                            "best_epoch", "epochs", "minutes"]])

FIG = Path("/content/figures"); FIG.mkdir(exist_ok=True)
curves = []
for run_id in RUN_IDS:
    fig = plot_learning_curves(run_dir(run_id) / "history.csv", FIG / f"curves_{run_id}.png", title=run_id)
    curves.append(fig)
    display(IPImage(str(fig)))

## 5 · Weakest classes on val (last run)

In [ ]:
per_class = pd.DataFrame(reports[RUN_IDS[-1]]["per_class"]).sort_values("f1")
per_class.head(8)[["class", "support", "precision", "recall", "f1"]]

## 6 · Download the results

`trained_runs.zip` holds `reports/results.csv`, `reports/runs/<run_id>/` (metrics, history, config, environment)
and `reports/figures/curves_<run_id>.png`. Send it in the chat; the models stay on Drive.

In [ ]:
from google.colab import files
from plantdx.results import export_for_repo

out = export_for_repo(RUN_IDS, "/content/trained_runs.zip", figures=curves)
files.download(str(out))